# Athena 01 — Create the `yelp_db` Database

This notebook creates the Glue/Athena database that will hold metadata for the raw Yelp review data uploaded by notebook `01_setup_S3_bucket.ipynb`.

Follows the same pattern as the Heart Valve `01_Create_Athena_Database.ipynb`.

In [ ]:
!pip install --disable-pip-version-check --quiet awswrangler PyAthena pandas

In [ ]:
import boto3
import sagemaker
import pandas as pd
from pyathena import connect

%store -r bucket
%store -r region
%store -r athena_staging_prefix

if "bucket" not in dir() or not bucket:
    account_id = boto3.client("sts").get_caller_identity()["Account"]
    bucket = f"yelp-sentiment-mlops-{account_id}"
    region = boto3.Session().region_name
    athena_staging_prefix = "athena/staging"

database_name = "yelp_db"
s3_staging_dir = f"s3://{bucket}/{athena_staging_prefix}/"

print("Bucket:           ", bucket)
print("Region:           ", region)
print("Athena staging:   ", s3_staging_dir)
print("Database to create:", database_name)

%store database_name
%store s3_staging_dir

In [ ]:
conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

statement = f"CREATE DATABASE IF NOT EXISTS {database_name}"
print(statement)
pd.read_sql(statement, conn)

In [ ]:
df_show = pd.read_sql("SHOW DATABASES", conn)
df_show

In [ ]:
ingest_create_athena_db_passed = database_name in df_show.values
%store ingest_create_athena_db_passed
print("Create Athena DB passed:", ingest_create_athena_db_passed)

## Done

Continue to `02_Register_S3_With_Athena.ipynb` to register the raw-reviews CSV as a queryable Athena table.